# <font color="#418FDE" size="6.5" uppercase>**MLP Training**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Configure MLP architecture, solvers, learning rates, and batch sizes. 
- Apply regularization, early stopping, warm starts, and incremental fitting. 
- Diagnose scaling, convergence, and practical limitations of scikit-learn MLPs. 


## **1. Architecture Regularization**

### **1.1. Alpha Regularization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_01_01.jpg?v=1784019110" width="250">



>* Alpha penalizes overly large MLP weights
>* Helps prevent memorization and improve generalization

>* Balance alpha to avoid underfitting or overfitting
>* Tune alpha based on data and model

>* Tune alpha with architecture and scaling
>* Validate alpha for reliable generalization



In [ ]:
#@title Python Code - Alpha Regularization

# Compare alpha values in one small MLP.
# Alpha penalizes overly large network weights.
# Validation accuracy shows the regularization tradeoff.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small noisy classification dataset.
features, target = make_moons(n_samples=600, noise=0.28, random_state=42)

# Check that the generated data has the expected shape.
if features.shape != (600, 2):
    raise ValueError("Expected 600 rows and 2 features.")

# Split before scaling to avoid data leakage.
X_train, X_valid, y_train, y_valid = train_test_split(
    features, target, test_size=0.35, stratify=target, random_state=42
)

alpha_values = [0.0001, 0.01, 0.1, 1.0, 10.0]
train_scores = []
valid_scores = []

# Train the same architecture with different alpha values.
for alpha in alpha_values:
    mlp = MLPClassifier(
        hidden_layer_sizes=(30,), alpha=alpha, max_iter=1200, random_state=42
    )

    model = make_pipeline(StandardScaler(), mlp)
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    valid_scores.append(model.score(X_valid, y_valid))

print("scikit-learn version:", sklearn.__version__)
print("alpha values:", alpha_values)
print("validation accuracy:", [round(score, 3) for score in valid_scores])

# Plot accuracy against alpha on a logarithmic scale.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alpha_values, train_scores, marker="o", label="training accuracy")
ax.plot(alpha_values, valid_scores, marker="o", label="validation accuracy")

ax.set_xscale("log")
ax.set_xlabel("alpha regularization strength")
ax.set_ylabel("accuracy")
ax.set_title("MLP alpha regularization tradeoff")
ax.legend()
plt.show()



### **1.2. Hidden Layer Design**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_01_02.jpg?v=1784019112" width="250">



>* Hidden layers control model learning capacity
>* Match architecture size to data complexity

>* Start simple, then expand if needed
>* Balance model power with reliable generalization

>* Tune architecture with training settings
>* Choose validated, stable, maintainable models



In [ ]:
#@title Python Code - Hidden Layer Design

# Compare hidden layer sizes for one MLP.
# Wider layers can increase model capacity.
# Validation accuracy reveals useful architecture tradeoffs.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small nonlinear classification dataset.
features, target = make_moons(n_samples=600, noise=0.25, random_state=42)

# Check that the generated data has the expected shape.
if features.shape != (600, 2):
    raise ValueError("Expected 600 rows and 2 features.")

# Split data before scaling to avoid leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Try compact architectures that differ only in hidden layers.
architectures = [(3,), (10,), (30,), (10, 10)]
labels = ["3 units", "10 units", "30 units", "10 + 10 units"]
accuracies = []

# Train one MLP per architecture using identical solver settings.
for hidden_layers in architectures:
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        solver="adam",
        learning_rate_init=0.01,
        batch_size=32,
        max_iter=500,
        random_state=42,
        verbose=False,
    )
    model = make_pipeline(StandardScaler(), mlp)
    model.fit(X_train, y_train)
    accuracies.append(model.score(X_test, y_test))

# Print concise results for the architecture comparison.
print("scikit-learn version:", sklearn.__version__)
print("Architecture comparison on held-out test data:")
for label, accuracy in zip(labels, accuracies):
    print(label + ": accuracy = " + str(round(accuracy, 3)))

# Plot accuracy so capacity differences are easy to compare.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, accuracies, color="steelblue")
ax.set_title("Hidden Layer Design and Test Accuracy")
ax.set_xlabel("MLP hidden_layer_sizes")
ax.set_ylabel("Test accuracy")
ax.set_ylim(0.7, 1.0)

plt.show()



### **1.3. Solver Choices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_01_03.jpg?v=1784019114" width="250">



>* lbfgs suits smaller, simpler MLP problems
>* Stochastic solvers scale better with batches

>* SGD offers control but needs careful tuning
>* Learning rate and batch size affect convergence

>* Adam is a strong general-purpose default
>* Match solver to data, model, and tuning



In [ ]:
#@title Python Code - Solver Choices

# Compare MLP solvers on scaled data.
# Solver choice changes training behavior.
# Results show accuracy and iterations.

import sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
wine = load_wine()
features = wine.data
target = wine.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split once so every solver sees identical data.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Keep the architecture fixed to isolate solver choice.
solver_names = ["lbfgs", "sgd", "adam"]
results = []

# Train one small MLP for each solver.
for solver_name in solver_names:
    mlp = MLPClassifier(
        hidden_layer_sizes=(12,), solver=solver_name, max_iter=800,
        random_state=42, learning_rate_init=0.01
    )

    model = make_pipeline(StandardScaler(), mlp)
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    iterations = model.named_steps["mlpclassifier"].n_iter_
    results.append((solver_name, accuracy, iterations))

print("scikit-learn version:", sklearn.__version__)
print("Same architecture: one hidden layer with 12 neurons.")
print("Solver | test accuracy | training iterations")

for solver_name, accuracy, iterations in results:
    print(f"{solver_name:5s} | {accuracy:.3f} | {iterations}")



## **2. Training Controls**

### **2.1. Learning Rate Control**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_02_01.jpg?v=1784019123" width="250">



>* Learning rate controls weight update size
>* Balanced steps improve stable, efficient learning

>* Adaptive rates refine learning as progress slows
>* Coordinate with regularization and early stopping

>* Scale features to stabilize MLP training
>* Tune learning rate with other controls



In [ ]:
#@title Python Code - Learning Rate Control

# Compare learning rates for MLP training.
# Scaling supports stable neural network updates.
# The plot shows faster or slower convergence.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
Y = data.target

# Validate the basic dataset shape before modeling.
if X.shape[0] != Y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so evaluation uses unseen examples.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.25, stratify=Y, random_state=42
)

# Try three learning rates with the same architecture.
learning_rates = [0.0005, 0.005, 0.05]
loss_curves = {}
test_scores = {}

for rate in learning_rates:
    mlp = MLPClassifier(
        hidden_layer_sizes=(20,),
        solver="adam",
        learning_rate_init=rate,
        alpha=0.001,
        max_iter=120,
        random_state=42,
        verbose=False,
    )

    model = make_pipeline(StandardScaler(), mlp)
    model.fit(X_train, Y_train)
    loss_curves[rate] = mlp.loss_curve_
    test_scores[rate] = model.score(X_test, Y_test)

# Print concise results for beginner comparison.
print("scikit-learn version:", sklearn.__version__)
for rate in learning_rates:
    score = round(test_scores[rate], 3)
    final_loss = round(loss_curves[rate][-1], 3)
    print("rate", rate, "test accuracy", score, "final loss", final_loss)

# Plot loss curves to show learning rate behavior.
fig, ax = plt.subplots(figsize=(7, 4))
for rate in learning_rates:
    ax.plot(loss_curves[rate], label="rate " + str(rate))

ax.set_title("MLP loss curves for different learning rates")
ax.set_xlabel("Training iteration")
ax.set_ylabel("Training loss")
ax.legend()

plt.show()



### **2.2. Batch Size**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_02_02.jpg?v=1784019121" width="250">



>* Small batches update often but noisily
>* Large batches update smoothly but slowly

>* Small batches add noise and regularization
>* Large batches steady early stopping signals

>* Balance batch size with data and resources
>* Monitor loss and validation for generalization



In [ ]:
#@title Python Code - Batch Size

# Compare batch sizes during MLP training.
# Smaller batches update weights more often.
# The plot shows validation accuracy differences.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so validation accuracy measures generalization.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Try three batch sizes with the same MLP settings.
batch_sizes = [8, 32, 128]
validation_scores = []

for batch_size in batch_sizes:
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=(20,),
            batch_size=batch_size,
            alpha=0.01,
            early_stopping=True,
            validation_fraction=0.2,
            n_iter_no_change=20,
            max_iter=300,
            random_state=42,
        ),
    )
    model.fit(X_train, y_train)
    validation_scores.append(model.score(X_valid, y_valid))

# Print concise context for the plotted comparison.
print("scikit-learn version:", sklearn.__version__)
print("Batch sizes tested:", batch_sizes)
print("Validation accuracies:", [round(score, 3) for score in validation_scores])

# Plot how batch size changes validation performance.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(batch_sizes, validation_scores, marker="o")
ax.set_title("MLP validation accuracy by batch size")
ax.set_xlabel("Batch size")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0.85, 1.0)
plt.show()



### **2.3. Early Stopping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_02_03.jpg?v=1784019125" width="250">



>* Prevents memorizing training data noise
>* Stops when validation performance stops improving

>* Balances underfitting and overfitting risks
>* Uses validation performance to guide stopping

>* Use representative validation data carefully
>* Coordinate with learning rate and regularization



In [ ]:
#@title Python Code - Early Stopping

# This example demonstrates early stopping in MLP training.
# Validation scores decide when training should stop.
# The plot shows improvement and automatic stopping.

import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split once for final testing after training.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Scale using only the training data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Configure early stopping with a validation split.
mlp = MLPClassifier(
    hidden_layer_sizes=(20,),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=8,
    tol=0.0001,
    random_state=42,
    verbose=False,
)

# Fit once and let validation performance stop training.
mlp.fit(X_train_scaled, y_train)
y_pred = mlp.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)

# Print concise results that explain the stopping behavior.
print("scikit-learn version:", sklearn.__version__)
print("Training iterations used:", mlp.n_iter_)
print("Final test accuracy:", round(test_accuracy, 3))

# Plot validation accuracy recorded during training.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(mlp.validation_scores_) + 1), mlp.validation_scores_)
ax.set_title("MLP Early Stopping Validation Scores")
ax.set_xlabel("Training iteration")
ax.set_ylabel("Validation accuracy")
ax.grid(True, alpha=0.3)
plt.show()



## **3. Practical MLP Limits**

### **3.1. Warm Start**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_03_01.jpg?v=1784019116" width="250">



>* Continue training from previously learned weights
>* Extend experiments without resetting model progress

>* Use warm starts to inspect convergence
>* Check scaling and solver settings first

>* Warm start cannot replace validation
>* Document use to compare experiments fairly



In [ ]:
#@title Python Code - Warm Start

# Warm start continues training from existing MLP weights.
# We compare repeated fits with and without warm start.
# The printed losses reveal practical convergence behavior.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Use a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split once so both models see identical data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.25, random_state=42
)

# Scaling is essential for practical MLP optimization.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Warm start keeps weights between separate fit calls.
warm_model = MLPClassifier(
    hidden_layer_sizes=(12,), max_iter=80, warm_start=True,
    random_state=42, solver="adam", learning_rate_init=0.001
)

# A normal model restarts from scratch on each fit call.
restart_model = MLPClassifier(
    hidden_layer_sizes=(12,), max_iter=80, warm_start=False,
    random_state=42, solver="adam", learning_rate_init=0.001
)

warm_losses = []
restart_losses = []

# Repeated fits show whether training is truly continuing.
for training_round in range(3):
    warm_model.fit(X_train_scaled, y_train)
    restart_model.fit(X_train_scaled, y_train)
    warm_losses.append(round(warm_model.loss_, 4))
    restart_losses.append(round(restart_model.loss_, 4))

warm_accuracy = round(warm_model.score(X_test_scaled, y_test), 3)
restart_accuracy = round(restart_model.score(X_test_scaled, y_test), 3)

print("scikit-learn version:", sklearn.__version__)
print("Warm-start losses after each fit:", warm_losses)
print("Restarted losses after each fit:", restart_losses)
print("Warm-start test accuracy:", warm_accuracy)
print("Restarted test accuracy:", restart_accuracy)



### **3.2. Incremental Fitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_03_02.jpg?v=1784019118" width="250">



>* Update MLPs gradually with new data batches
>* Avoid costly full retraining for changing data

>* Batch order can bias incremental learning
>* Monitor validation, balance, scaling, and forgetting

>* Keep scaling consistent across incremental updates
>* Monitor convergence and know retraining limits



In [ ]:
#@title Python Code - Incremental Fitting

# Demonstrate incremental fitting for a small MLP.
# Track validation accuracy after each mini batch.
# Show why monitoring updates matters in practice.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# Create a small classification dataset for fast training.
features, target = make_classification(
    n_samples=900,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_classes=3,
    random_state=42,
)

# Keep a stable test set for monitoring every update.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.25,
    stratify=target,
    random_state=42,
)

# Fit scaling only on training data to avoid leakage.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)
test_scaled = scaler.transform(test_features)

# Validate the basic shapes before incremental training.
if train_scaled.shape[0] != train_target.shape[0]:
    raise ValueError("Training features and labels must match.")

# Shuffle once so each mini batch is more representative.
rng = np.random.default_rng(42)
order = rng.permutation(train_scaled.shape[0])
train_scaled = train_scaled[order]
train_target = train_target[order]

# Configure one small MLP for repeated partial_fit calls.
mlp = MLPClassifier(
    hidden_layer_sizes=(20,),
    solver="sgd",
    learning_rate_init=0.02,
    alpha=0.001,
    random_state=42,
    max_iter=1,
)

# Update the model batch by batch and record accuracy.
classes = np.unique(train_target)
batch_size = 75
accuracies = []

for start in range(0, train_scaled.shape[0], batch_size):
    stop = start + batch_size
    batch_features = train_scaled[start:stop]
    batch_target = train_target[start:stop]
    mlp.partial_fit(batch_features, batch_target, classes=classes)
    predictions = mlp.predict(test_scaled)
    accuracies.append(accuracy_score(test_target, predictions))

# Print concise results that connect updates to monitoring.
print("scikit-learn version:", sklearn.__version__)
print("Number of incremental updates:", len(accuracies))
print("First test accuracy:", round(accuracies[0], 3))
print("Best test accuracy:", round(max(accuracies), 3))
print("Final test accuracy:", round(accuracies[-1], 3))

# Plot validation accuracy after each incremental update.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(accuracies) + 1), accuracies, marker="o")
ax.set_title("MLP partial_fit monitoring")
ax.set_xlabel("Incremental update number")
ax.set_ylabel("Test accuracy")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.show()



### **3.3. Scaling Convergence**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_21/Lecture_B/image_03_03.jpg?v=1784019120" width="250">



>* MLPs need consistently scaled input features
>* Poor scaling can disrupt convergence

>* Convergence warnings signal training needs inspection
>* Check scaling, learning rate, and model design

>* Best for modest tabular neural network tasks
>* Check convergence before trusting deployment performance



In [ ]:
#@title Python Code - Scaling Convergence

# This example shows why scaling affects MLP convergence.
# We compare identical models with and without scaling.
# The scaled model should converge more smoothly.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small classification dataset with uneven feature scales.
features, target = make_classification(
    n_samples=900,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    random_state=42,
)

# Stretch two columns to imitate mixed real-world measurement units.
features[:, 0] = features[:, 0] * 1000
features[:, 1] = features[:, 1] * 0.01

# Check that the dataset has the expected beginner-friendly shape.
if features.shape != (900, 8):
    raise ValueError("Unexpected dataset shape for this lesson.")

# Split before scaling to avoid leaking test information.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.25,
    stratify=target,
    random_state=42,
)

# Use the same small MLP settings in both comparisons.
mlp_settings = dict(
    hidden_layer_sizes=(20,),
    solver="adam",
    learning_rate_init=0.01,
    max_iter=120,
    random_state=42,
    verbose=False,
)

# Train one MLP directly on the uneven raw feature scales.
raw_model = MLPClassifier(**mlp_settings)
raw_model.fit(train_features, train_target)

# Train the same MLP after fitting scaling only on training data.
scaled_model = make_pipeline(StandardScaler(), MLPClassifier(**mlp_settings))
scaled_model.fit(train_features, train_target)

# Collect accuracy and iteration counts for a concise comparison.
raw_accuracy = raw_model.score(test_features, test_target)
scaled_accuracy = scaled_model.score(test_features, test_target)
scaled_mlp = scaled_model.named_steps["mlpclassifier"]

print("scikit-learn version:", sklearn.__version__)
print("Raw features accuracy:", round(raw_accuracy, 3))
print("Scaled features accuracy:", round(scaled_accuracy, 3))
print("Raw iterations used:", raw_model.n_iter_)
print("Scaled iterations used:", scaled_mlp.n_iter_)

# Plot the loss curves to show convergence behavior directly.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(raw_model.loss_curve_, label="Raw feature scales")
ax.plot(scaled_mlp.loss_curve_, label="Standardized features")
ax.set_title("MLP Loss Curves With and Without Scaling")
ax.set_xlabel("Training iteration")
ax.set_ylabel("Training loss")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**MLP Training**</font>


In this lecture, you learned to:
- Configure MLP architecture, solvers, learning rates, and batch sizes. 
- Apply regularization, early stopping, warm starts, and incremental fitting. 
- Diagnose scaling, convergence, and practical limitations of scikit-learn MLPs. 

In the next Module (Module 22), we will go over 'Clustering Foundations'